In [ ]:
"""Calculate Win Expectancy and Leverage Index"""
import pandas as pd
import numpy as np
import pickle
from typing import Dict, Tuple

# STEP 1: Load Your Trackman Data
query = """WITH startTimes AS (
    SELECT 
        gameUID
        , localDateTime
    FROM [base].[mitts_trackman_play]
    WHERE pitchNo = 1
    GROUP BY gameUID, localDateTime),
endTimes AS (
    SELECT 
        gameUID
        , MAX(localDateTime) AS localDateTime
    FROM [base].[mitts_trackman_play]
    GROUP BY gameUID)
SELECT
    PLAY.playID
    , PLAY.gameUID
    , PLAY.localDateTime
    , startTimes.localDateTime AS startTime
    , endTimes.localDateTime AS endTime
    , CASE WHEN DATEPART(HOUR, PLAY.localDateTime) < 5 THEN DATEADD(DAY, -1, PLAY.localDateTime) ELSE PLAY.localDateTime END AS dateNew
    , EBE.date
    , PLAY.batterTeam
    , PLAY.pitcherTeam
    , pitchCall
    , playResult
    , kOrBB
    , balls
    , strikes
    , outs
    , plateLocHeight
    , plateLocSide
    , in_zone
    , zone_loc
    , hit_type_classification
    , in_play
    , r_1B
    , r_2B
    , r_3B
    , PLAY.inning
    , PLAY.PAofInning
    , EBE.PAofInning
    , PLAY.topBottom
    , outsOnPlay
    , runsScored
    , batterName
    , pitcherName
    , pitchofPA
    , max_event_of_PA
    , event_of_PA
    , r1_name
    , r2_name
    , r3_name
    , description
    , SESS.leagueShortName
      
FROM [base].[mitts_trackman_play] PLAY
JOIN [base].[mitts_trackman_pitch] PITCH ON PITCH.playID = PLAY.playID
JOIN [base].[mitts_trackman_session] SESS ON SESS.gameUID = PLAY.gameUID
JOIN [base].[vw_mitts_pbp_outcomes] pbp ON pbp.playID = PLAY.playID
JOIN startTimes ON startTimes.gameUID = PLAY.gameUID 
JOIN endTimes ON endTimes.gameUID = PLAY.gameUID
JOIN [base].[scraped_baserunner_ebe] EBE ON PLAY.batterTeam = EBE.batterTeam AND PLAY.pitcherTeam = EBE.pitcherTeam AND PLAY.inning = EBE.inning
                AND PLAY.PAofInning = EBE.PAofInning AND PLAY.topBottom = EBE.topBottom AND YEAR(EBE.date) = YEAR(PLAY.localDateTime) AND MONTH(EBE.date) = MONTH(PLAY.localDateTime)
                AND EBE.date = CASE 
                    WHEN CAST(PLAY.localDateTime AS DATE) != CAST(startTimes.localDateTime AS DATE) 
                    THEN CAST(startTimes.localDateTime AS DATE)
                    ELSE CAST(PLAY.localDateTime AS DATE) 
                END             
WHERE  ((pitchofPA > 1 AND event_of_PA = max_event_of_PA) OR (pitchofPA = 1 AND event_of_PA = 1))
AND (YEAR(PLAY.localDateTime) IN (2023,2024, 2025))  
AND (PLAY.batterTeam IN (SELECT teamName FROM [base].[vw_mitts_power_5_plus]) AND PLAY.pitcherTeam IN (SELECT teamName FROM [base].[vw_mitts_power_5_plus]))
AND (PLAY.localDateTime >= startTimes.localDateTime) AND (PLAY.localDateTime <= endTimes.localDateTime)
ORDER BY PLAY.gameUID, PLAY.localDateTime"""

# Create database connection and run query
#User and Password hidden from public view

df = db_conn.run_query (query, file=False)
print(f"✓ Loaded {len(df)} plate appearances")
print(f"df type: {type(df)}")  # Should say DataFrame
print(f"df is None: {df is None}")  # Should say False

In [ ]:
# STEP 1: Define the FIXED function
def calculate_scores_before_play(df):
    """Calculate running score BEFORE each play (not after)"""    
    # Sort chronologically within each game
    df = df.sort_values(['gameUID', 'localDateTime']).reset_index(drop=True)
    
    # Create columns for runs scored by each team on this play
    df['hometeam_runs_this_play'] = df.apply(lambda x: x['runsScored'] if x['topBottom'] == 'bottom' else 0, axis=1)
    df['awayteam_runs_this_play'] = df.apply(lambda x: x['runsScored'] if x['topBottom'] == 'top' else 0, axis=1)
    
    # Calculate cumulative score AFTER each play
    df['home_score_after'] = df.groupby('gameUID')['hometeam_runs_this_play'].cumsum()
    df['away_score_after'] = df.groupby('gameUID')['awayteam_runs_this_play'].cumsum()
    
    # Shift to get score BEFORE the play
    df['home_score'] = df.groupby('gameUID')['home_score_after'].shift(1).fillna(0)
    df['away_score'] = df.groupby('gameUID')['away_score_after'].shift(1).fillna(0)
    
    # Calculate score differential
    df['score_diff'] = df['home_score'] - df['away_score']
    
    # Drop temporary columns
    df = df.drop(['hometeam_runs_this_play', 'awayteam_runs_this_play', 'home_score_after', 'away_score_after'], axis=1)
    return df

# STEP 2: CALL the function to update df
df = calculate_scores_before_play(df)

# STEP 3: Verify it worked
print(f"home_score sum: {df['home_score'].sum()}")
print(f"away_score sum: {df['away_score'].sum()}")

df_sample = df[['gameUID', 'inning', 'topBottom', 'home_score', 'away_score', 'score_diff', 'runsScored']]
df_sample.head(50)

In [ ]:
# ========================================
# STEP 3: Determine Game Winners
# ========================================
def add_game_winners(df):
    """Add the final winner for each game. Uses each play's score AFTER that play, so runs on the final play of a game are included in the final score."""
    df = df.copy()

    # Remove winner columns left from prior runs of this notebook.
    winner_columns_to_remove = [
        column
        for column in df.columns
        if (column in ['home_final_score','away_final_score','home_wins','away_wins']
            or column.startswith('home_wins_') or column.startswith('away_wins_')
            or column.startswith('home_final_score_') or column.startswith('away_final_score_'))]

    df = df.drop(columns=winner_columns_to_remove, errors='ignore')

    runs_scored = pd.to_numeric(df['runsScored'],errors='coerce').fillna(0)

    # Score immediately after each play.
    df['_home_score_after_play'] = (df['home_score'] + np.where(df['topBottom'].eq('bottom'), runs_scored, 0))
    df['_away_score_after_play'] = ( df['away_score'] + np.where(df['topBottom'].eq('top'), runs_scored, 0))

    # Final score is the maximum post-play score for each team.
    final_scores = (df.groupby('gameUID', as_index=False).agg(home_final_score=('_home_score_after_play', 'max'), away_final_score=('_away_score_after_play', 'max')))
    final_scores['home_wins'] = (final_scores['home_final_score'] > final_scores['away_final_score']).astype(int)
    final_scores['away_wins'] = (final_scores['away_final_score'] > final_scores['home_final_score']).astype(int)

    df = df.merge(
        final_scores[['gameUID', 'home_final_score','away_final_score', 'home_wins', 'away_wins']], on='gameUID', how='left', validate='many_to_one')

    df = df.drop(columns=['_home_score_after_play','_away_score_after_play'])

    print(f"Home wins: {final_scores['home_wins'].sum():,}")
    print(f"Away wins: {final_scores['away_wins'].sum():,}")

    return df
df = add_game_winners(df)

winner_sample = df[['gameUID', 'inning', 'topBottom', 'home_score', 'away_score', 'home_final_score','away_final_score','home_wins','away_wins']]
winner_sample.head(50)

In [ ]:
# ========================================
# STEP 4: Add Base State
# ========================================
def add_base_state(df):
    """Encode base runner state as string"""
    def encode_bases(row):
        if row['r_1B'] == 0 and row['r_2B'] == 0 and row['r_3B'] == 0: return '---'
        elif row['r_1B'] == 1 and row['r_2B'] == 0 and row['r_3B'] == 0: return '1--'
        elif row['r_1B'] == 0 and row['r_2B'] == 1 and row['r_3B'] == 0: return '-2-'
        elif row['r_1B'] == 0 and row['r_2B'] == 0 and row['r_3B'] == 1: return '--3'
        elif row['r_1B'] == 1 and row['r_2B'] == 1 and row['r_3B'] == 0: return '12-'
        elif row['r_1B'] == 1 and row['r_2B'] == 0 and row['r_3B'] == 1: return '1-3'
        elif row['r_1B'] == 0 and row['r_2B'] == 1 and row['r_3B'] == 1: return '-23'
        elif row['r_1B'] == 1 and row['r_2B'] == 1 and row['r_3B'] == 1: return '123'
        else: return '---'
    
    df['base_state'] = df.apply(encode_bases, axis=1)
    print(df['base_state'].value_counts().sort_index())
    return df

df = add_base_state(df)

# Create sample for viewing in VSCode
base_state_sample = df[['gameUID', 'inning', 'outs', 'base_state', 'r_1B', 'r_2B', 'r_3B']]
base_state_sample.head(50)

In [ ]:
# ========================================
# STEP 5: Add Home-Team Win Outcome
# ========================================

def add_home_team_win_outcome(df):
    """Define the outcome used for win expectancy.Every WE value means the same thing: probability that the HOME team wins."""
    df = df.copy() 
    df['home_team_won'] = df['home_wins'].astype(int)

    print(df['home_team_won'].value_counts().sort_index())
    return df

df = add_home_team_win_outcome(df)

home_win_sample = df[['gameUID','inning', 'topBottom', 'home_score', 'away_score', 'home_wins', 'away_wins', 'home_team_won' ]]
home_win_sample.head(50)

In [ ]:
# STEP 6: Win Expectancy

from itertools import product

def build_smoothed_win_expectancy_table(df, state_prior_strength=50, inning_prior_strength=100, context_prior_strength=50):
    df = df.copy()
    
    df['score_diff_capped'] = pd.to_numeric(df['score_diff'], errors='coerce').fillna(0).clip(-7, 7).astype(int)
    df['inning'] = pd.to_numeric(df['inning'], errors='coerce').astype(int)
    df['inning_bucket'] = df['inning'].clip(upper=10)
    df['outs'] = pd.to_numeric(df['outs'], errors='coerce').astype(int)
    df['home_team_won'] = pd.to_numeric(df['home_team_won'], errors='coerce')

    state_cols = ['inning_bucket', 'topBottom', 'score_diff_capped', 'outs', 'base_state']
    parent_cols = ['inning_bucket', 'topBottom', 'score_diff_capped']
    score_cols = ['topBottom', 'score_diff_capped']
    context_cols = ['topBottom', 'score_diff_capped', 'outs', 'base_state']

    observed = (df.groupby(state_cols, as_index=False).agg(wins=('home_team_won', 'sum'), total=('home_team_won', 'count')))

    global_we = observed['wins'].sum() / observed['total'].sum()

    half_score = (observed.groupby(score_cols, as_index=False).agg(half_score_wins=('wins', 'sum'), half_score_total=('total', 'sum')))
    half_score['half_score_we'] = half_score['half_score_wins'] / half_score['half_score_total']

    inning_score = (observed.groupby(parent_cols, as_index=False).agg(inning_score_wins=('wins', 'sum'), inning_score_total=('total', 'sum')))
    inning_score = inning_score.merge(half_score[score_cols + ['half_score_we']], on=score_cols, how='left')
    inning_score['half_score_we'] = inning_score['half_score_we'].fillna(global_we)
    inning_score['inning_score_we'] = (inning_score['inning_score_wins'] + inning_prior_strength * inning_score['half_score_we']) / (inning_score['inning_score_total'] + inning_prior_strength)

    context = (observed.groupby(context_cols, as_index=False).agg(context_wins=('wins', 'sum'), context_total=('total', 'sum')))
    context = context.merge(half_score[score_cols + ['half_score_we']],on=score_cols,how='left')
    context['half_score_we'] = context['half_score_we'].fillna(global_we)
    context['context_we'] = (context['context_wins'] + context_prior_strength * context['half_score_we']) / (context['context_total'] + context_prior_strength)
    context['base_out_adjustment'] = context['context_we'] - context['half_score_we']

    innings = list(range(1, 11))
    halves = sorted(df['topBottom'].dropna().unique())
    score_diffs = list(range(-7, 8))
    outs_values = [0, 1, 2]
    base_states = ['---', '1--', '-2-', '--3', '12-', '1-3', '-23', '123']

    we_table = pd.DataFrame(list(product(innings, halves, score_diffs, outs_values, base_states)),columns=state_cols)

    we_table = we_table.merge(observed, on=state_cols, how='left')
    we_table['wins'] = we_table['wins'].fillna(0)
    we_table['total'] = we_table['total'].fillna(0).astype(int)
    we_table = we_table.merge(inning_score[parent_cols + ['inning_score_we']], on=parent_cols,how='left')
    we_table = we_table.merge(half_score[score_cols + ['half_score_we']],on=score_cols,how='left')
    we_table = we_table.merge( context[context_cols + ['base_out_adjustment']],on=context_cols,how='left')
    we_table['baseline_we'] = ( we_table['inning_score_we'].fillna(we_table['half_score_we']).fillna(global_we))
    we_table['prior_we'] = ( we_table['baseline_we'] + we_table['base_out_adjustment'].fillna(0)).clip(0.001, 1)
    we_table['win_expectancy'] = (we_table['wins'] + state_prior_strength * we_table['prior_we']) / (we_table['total'] + state_prior_strength)
    we_table['win_expectancy'] = we_table['win_expectancy'].clip(0.001, 1)
    we_table = we_table.rename(columns={'inning_bucket': 'inning','score_diff_capped': 'score_diff'})

    return we_table[['inning', 'topBottom', 'score_diff', 'outs', 'base_state', 'wins','prior_we', 'win_expectancy']].copy()

df['score_diff_capped'] = pd.to_numeric(df['score_diff'], errors='coerce').fillna(0).clip(-7, 7).astype(int)
df['inning_bucket'] = pd.to_numeric(df['inning'], errors='coerce').clip(upper=10).astype(int)

we_table = build_smoothed_win_expectancy_table(df)
we_table.head(25)

In [ ]:
# ========================================
# STEP 7: Calculate Event Probabilities (Boss's Instructions)
# ========================================
def calculate_event_probabilities(df):
    """Calculate probability of each outcome
    1. Filter to EVENT-level (not pitch-level): Remove rows where BOTH playResult = 'undefined' AND kOrBB is undefined
    2. Separate strikeouts from other outs
    3. Get walks from kOrBB column"""
    
    # Step 1: Filter to EVENT-level only. Keep rows where EITHER playResult is defined OR kOrBB is defined
    df_events = df[~((df['playResult'] == 'undefined') & (df['kOrBB'].isna()))].copy()

    # Step 2: Map outcomes - kOrBB for K/BB, playResult for hits/outs
    def map_outcome(row):
        korbb = str(row['kOrBB']).lower() if pd.notna(row['kOrBB']) else ''
        play = str(row['playResult']).lower() if pd.notna(row['playResult']) else ''
        
        # Priority 1: Check kOrBB for strikeouts and walks
        if korbb in ['k', 'strikeout']: return 'K'
        elif korbb in ['bb', 'walk']: return 'walk'
        elif korbb == 'hbp': return 'walk'
        
        # Priority 2: Check playResult for hits and other outs
        elif 'single' in play: return '1B'
        elif 'double' in play: return '2B'
        elif 'triple' in play: return '3B'
        elif 'homerun' in play: return 'HR'
        elif play in ['out', 'fielderschoice', 'sacrifice', 'groundout','flyout', 'lineout', 'popout', 'error', 'caughtstealing']: return 'out'
        elif 'stolenbase' in play: return None  # Ignore stolen bases
        else: return None
    
    df_events['outcome'] = df_events.apply(map_outcome, axis=1)
    # Remove rows with no outcome mapped
    df_with_outcomes = df_events[df_events['outcome'].notna()].copy()
    # Calculate probabilities
    total_pa = len(df_with_outcomes)
    outcome_counts = df_with_outcomes['outcome'].value_counts()
    event_probs = {'K': outcome_counts.get('K', 0) / total_pa,
        'out': outcome_counts.get('out', 0) / total_pa,
        'walk': outcome_counts.get('walk', 0) / total_pa,
        '1B': outcome_counts.get('1B', 0) / total_pa,
        '2B': outcome_counts.get('2B', 0) / total_pa,
        '3B': outcome_counts.get('3B', 0) / total_pa,
        'HR': outcome_counts.get('HR', 0) / total_pa}
    
    for outcome, prob in event_probs.items():
        print('Event Probabilites:' f"    {outcome}: {prob:.3f}")
    
    total_prob = sum(event_probs.values())
    return event_probs, df_with_outcomes

event_probs, df = calculate_event_probabilities(df)
outcome_sample = df[['gameUID', 'inning', 'topBottom', 'batterName', 'playResult', 'kOrBB', 'outcome']]
outcome_sample.head(50)

In [ ]:
# ========================================
# STEP 8: Calculate Leverage Index
# ========================================
def build_we_lookup(we_table):
    lookup_table = we_table.copy()
    lookup_table['score_diff'] = lookup_table['score_diff'].clip(-7, 7)
    return lookup_table.set_index(['inning', 'topBottom', 'score_diff', 'outs', 'base_state'])['win_expectancy'].to_dict()

missing_we_keys = set()

def lookup_we(we_lookup, inning, top_bottom, score_diff, outs, base_state):
    key = (int(min(inning, 10)), top_bottom,int(np.clip(score_diff, -7, 7)),int(outs),base_state)
    we = we_lookup.get(key, np.nan)
    if pd.isna(we): missing_we_keys.add(key)
    return we

def advance_runners_simple(base_state, outcome, outs):
    runners = {'1B': '1' in base_state,'2B': '2' in base_state,'3B': '3' in base_state}
    if outcome == 'HR':return '---', sum(runners.values()) + 1, outs
    if outcome == '3B':return '--3', sum(runners.values()), outs
    if outcome == '2B':
        runs = int(runners['2B']) + int(runners['3B'])
        new_bases = '-23' if runners['1B'] else '-2-'
        return new_bases, runs, outs
    if outcome == '1B':
        runs = int(runners['2B']) + int(runners['3B'])
        new_bases = '12-' if runners['1B'] else '1--'
        return new_bases, runs, outs
    if outcome == 'walk':
        if runners['1B'] and runners['2B'] and runners['3B']:return '123', 1, outs
        if runners['1B'] and runners['2B']: return '123', 0, outs
        if runners['1B'] and runners['3B']:return '1-3', 0, outs
        if runners['2B'] and runners['3B']:return '123', 0, outs
        if runners['1B']: return '12-', 0, outs
        if runners['2B']:return '12-', 0, outs
        if runners['3B']:return '1-3', 0, outs
        return '1--', 0, outs
    if outcome in ['out', 'K']:
        new_outs = outs + 1
        if new_outs >= 3: return '---', 0, 3
        return base_state, 0, new_outs
    return base_state, 0, outs

def terminal_home_we(inning,top_bottom,score_diff_after,outs_after,runs_scored):
    """Return 1.0 or 0.0 when the play ends the game.score_diff_after = home score minus away score."""
    if inning < 9:return None
    # Walk-off in the bottom of the 9th or later
    if (top_bottom == 'bottom' and runs_scored > 0 and score_diff_after > 0): return 1.0
    # Three outs during the top half only ends the game if the home team is ahead.
    if outs_after < 3:return None
    if top_bottom == 'top': return 1.0 if score_diff_after > 0 else None
    # Three outs in the bottom half ends the game whenever the score is not tied.
    if top_bottom == 'bottom' and score_diff_after != 0: return 1.0 if score_diff_after > 0 else 0.0
    return None

def calculate_leverage_index_for_state(inning,top_bottom,score_diff,outs,base_state,we_lookup,event_probs):
    current_we = lookup_we( we_lookup, inning, top_bottom,score_diff, outs,base_state)

    if pd.isna(current_we): return np.nan
    expected_swing = 0.0
    for outcome in ['K', 'out', 'walk', '1B', '2B', '3B', 'HR']:
        new_bases, runs_scored, outs_after = advance_runners_simple( base_state, outcome, outs)
        # Update score differential from the home team's perspective
        if top_bottom == 'top':new_score_diff = score_diff - runs_scored
        else:new_score_diff = score_diff + runs_scored

        # Check whether this simulated outcome ends the game
        new_we = terminal_home_we(inning, top_bottom, new_score_diff, outs_after,runs_scored)

        if new_we is None:
            # Move to the next half-inning if three outs occur
            if outs_after >= 3:

                if top_bottom == 'top':
                    new_inning = inning
                    new_top_bottom = 'bottom'

                else:
                    new_inning = min(inning + 1, 10)
                    new_top_bottom = 'top'
                new_bases = '---'
                new_outs = 0

            else:
                new_inning = inning
                new_top_bottom = top_bottom
                new_outs = outs_after

            new_we = lookup_we(we_lookup, new_inning, new_top_bottom, new_score_diff, new_outs, new_bases)

        if pd.isna(new_we):return np.nan
        expected_swing += (event_probs.get(outcome, 0)* abs(new_we - current_we))
        
    return expected_swing

# Build Leverage Index Dataset
li_df = df.copy()
li_df = li_df.drop(columns=['leverage_swing', 'leverage_index'],errors='ignore')

# Keep only event-level rows
valid_mask = pd.Series(True, index=li_df.index)
if 'playResult' in li_df.columns and 'kOrBB' in li_df.columns:
    valid_mask &= (li_df['playResult'].fillna('undefined').ne('undefined')
        | li_df['kOrBB'].fillna('undefined').ne('undefined'))
elif 'playResult' in li_df.columns: valid_mask &= (li_df['playResult'].fillna('undefined').ne('undefined'))
elif 'kOrBB' in li_df.columns:valid_mask &= (li_df['kOrBB'].fillna('undefined').ne('undefined'))

li_df = li_df.loc[valid_mask].copy()
li_df['_source_index'] = li_df.index

# Create WE Lookup
state_cols = ['inning_bucket','topBottom','score_diff','outs','base_state']
we_lookup = build_we_lookup(we_table)

# Calculate Unique-State Leverage Swings
unique_states = (li_df[state_cols].drop_duplicates().reset_index(drop=True))

unique_states['leverage_swing'] = unique_states.apply(
    lambda row: calculate_leverage_index_for_state(row['inning_bucket'], row['topBottom'], row['score_diff'], row['outs'], row['base_state'], we_lookup,event_probs),axis=1)

# Merge Leverage Swing Back to Events
li_df = li_df.merge(unique_states[state_cols + ['leverage_swing']],on=state_cols,how='left',validate='many_to_one')
#Calculate Leverage Index
avg_swing = li_df['leverage_swing'].mean()
if pd.isna(avg_swing) or avg_swing == 0:raise ValueError("Average leverage swing could not be calculated.")
li_df['leverage_index'] = (li_df['leverage_swing'] / avg_swing)

df['leverage_swing'] = np.nan
df['leverage_index'] = np.nan
df.loc[li_df['_source_index'],'leverage_swing'] = li_df['leverage_swing'].to_numpy()
df.loc[li_df['_source_index'],'leverage_index'] = li_df['leverage_index'].to_numpy()

print(f"Mean LI: {li_df['leverage_index'].mean():.3f}")
print(f"Min LI: {li_df['leverage_index'].min():.3f}")
print(f"Max LI: {li_df['leverage_index'].max():.3f}")

li_df

In [ ]:
# STEP 9: Validate Highest-Leverage States

validation_cols = ['inning','inning_bucket','topBottom','score_diff_capped','outs','base_state','leverage_swing','leverage_index']

top_li_states = (df.loc[df['leverage_index'].notna(),validation_cols]
    .drop_duplicates().sort_values('leverage_index', ascending=False).head(30).copy())

we_details = we_table [['inning','topBottom','score_diff','outs','base_state','wins','win_expectancy']].rename(columns={'inning': 'inning_bucket','score_diff': 'score_diff_capped'})

top_li_validation = top_li_states.merge(we_details,
    on=['inning_bucket','topBottom','score_diff_capped','outs','base_state'],how='left')
top_li_validation = top_li_validation[['inning','inning_bucket','topBottom','score_diff_capped','outs','base_state','leverage_swing','leverage_index','wins','win_expectancy']]
print("Highest-LI states with WE sample sizes:")
print(top_li_validation.to_string(index=False))
print("\nWE sample-size distribution:")
print("\nStates with fewer than 10 observations:")
print("\nLI states skipped because at least one required WE state was missing:")
print(unique_states['leverage_swing'].isna().sum())

In [ ]:
# STEP 12: LI Distribution, Half-Inning Average, and WE vs. LI
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = df.loc[df['leverage_index'].notna()].copy()
plot_df['score_diff_capped'] = plot_df['score_diff'].clip(-7, 7).astype(int)
plot_df = plot_df.drop(columns='win_expectancy', errors='ignore')

we_plot = we_table[['inning','topBottom','score_diff','outs','base_state','win_expectancy']].copy()
we_plot['inning_bucket'] = we_plot['inning']
plot_df = plot_df.merge(we_plot,left_on=['inning_bucket','topBottom','score_diff_capped','outs','base_state'],
    right_on=['inning_bucket','topBottom','score_diff','outs','base_state'],how='left',validate='many_to_one')

half_inning_summary = (plot_df.groupby(['inning_bucket','topBottom'], as_index=False).agg(avg_li=('leverage_index','mean'), n_events=('leverage_index','size')))

half_inning_plot = half_inning_summary[half_inning_summary['n_events'] >= 100].copy()

sns.set_theme(style='whitegrid', context='notebook')
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

sns.histplot(data=plot_df, x='leverage_index', bins=50, color='#1f77b4', edgecolor='white', ax=axes[0])
axes[0].axvline(1, color='crimson', linestyle='--', linewidth=2, label='Average LI = 1')
axes[0].set_title('Leverage Index Distribution')
axes[0].set_xlabel('Leverage Index')
axes[0].set_ylabel('Events')
axes[0].legend()

sns.lineplot(data=half_inning_plot, x='inning_bucket', y='avg_li', hue='topBottom',marker='o', linewidth=2.5, palette={'top':'#1f77b4','bottom':'#ff7f0e'}, ax=axes[1])
axes[1].axhline(1, color='crimson', linestyle='--', linewidth=2, label='Average LI = 1')
axes[1].set_title('Average LI by Half-Inning\n(100+ Events)')
axes[1].set_xlabel('Inning')
axes[1].set_ylabel('Average Leverage Index')
axes[1].legend(title='Half')

sns.scatterplot(data=plot_df, x='win_expectancy', y='leverage_index',alpha=0.08, s=12, color='#2ca02c', linewidth=0, ax=axes[2])
axes[2].axhline(1, color='crimson', linestyle='--', linewidth=2, label='Average LI = 1')
axes[2].set_title('Win Expectancy vs. Leverage Index')
axes[2].set_xlabel('Home-Team Win Expectancy')
axes[2].set_ylabel('Leverage Index')
axes[2].legend()

plt.tight_layout()
plt.show()

display(half_inning_plot.sort_values(['inning_bucket','topBottom']).style.format({'avg_li':'{:.3f}','n_events':'{:,.0f}'}))

In [ ]:
# STEP 13: Average LI by Inning and Score Differential

li_source = df.loc[df['leverage_index'].notna()].copy()
li_source['score_diff_capped'] = li_source['score_diff'].clip(-7, 7).astype(int)

heat_summary = (li_source.groupby(['topBottom', 'inning_bucket', 'score_diff_capped'], as_index=False)
    .agg(avg_li=('leverage_index', 'mean'), n_pa=('leverage_index', 'size')))

heat_summary = heat_summary.loc[heat_summary['n_pa'] >= 100].copy()
vmax = heat_summary['avg_li'].quantile(0.95)

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

for ax, half in zip(axes, ['top', 'bottom']):
    heat_data = heat_summary.loc[heat_summary['topBottom'].eq(half)]
    heatmap = heat_data.pivot(index='score_diff_capped', columns='inning_bucket', values='avg_li')

    sns.heatmap(heatmap.sort_index(ascending=False),cmap='YlOrRd',vmin=0,vmax=vmax,annot=True,fmt='.2f',linewidths=0.4,linecolor='white',cbar=ax is axes[1],cbar_kws={'label': 'Average LI'}, ax=ax)

    ax.set_title(f'Average LI: {half.title()} of Inning')
    ax.set_xlabel('Inning')
    ax.set_ylabel('Home Score Differential\n(Home − Away)')

plt.tight_layout()
plt.show()

In [ ]:
# STEP 14: Average LI by Base State and Outs
base_order = ['---', '1--', '-2-', '--3', '12-', '1-3', '-23', '123']

base_out_summary = (
    df.loc[df['leverage_index'].notna()]
    .groupby(['base_state', 'outs'], as_index=False)
    .agg(avg_li=('leverage_index', 'mean'), n_pa=('leverage_index', 'size'))
)

base_out_summary = base_out_summary.loc[base_out_summary['n_pa'] >= 100].copy()

li_matrix = (
    base_out_summary
    .pivot(index='base_state', columns='outs', values='avg_li')
    .reindex(base_order)
)

count_matrix = (
    base_out_summary
    .pivot(index='base_state', columns='outs', values='n_pa')
    .reindex(base_order)
)

labels = li_matrix.copy().astype(object)

for base in li_matrix.index:
    for outs in li_matrix.columns:
        li_value = li_matrix.loc[base, outs]
        count = count_matrix.loc[base, outs]

        labels.loc[base, outs] = (
            '' if pd.isna(li_value)
            else f'{li_value:.2f}\n(n={count:,.0f})'
        )

plt.figure(figsize=(9, 7))

sns.heatmap(
    li_matrix,
    cmap='YlOrRd',
    vmin=0,
    vmax=li_matrix.stack().quantile(0.95),
    annot=labels,
    fmt='',
    linewidths=0.4,
    linecolor='white',
    cbar_kws={'label': 'Average LI'}
)

plt.title('Average Leverage Index by Base State and Outs')
plt.xlabel('Outs')
plt.ylabel('Base State')
plt.tight_layout()
plt.show()